<div align="right"><sub>Notebook 最終更新: </sub></div>
<h1><strong>07. LangGraph による Executor-Critic エージェント（参照用）</strong></h1>

このノートブックは **06回** で実装した Executor-Critic パターンを，**LangGraph** を使って書き直した参照用資料です。

06回の手作りループ (`for` + `break`) と LangGraph の **StateGraph** を比較することで，フレームワークが何を抽象化しているかを理解することが目的です。

---

### 06回との対応関係

| 06回（手作り） | 07回（LangGraph） |
|---|---|
| `LLMExecutorCriticAgent` クラス | `StateGraph` + ノード関数 |
| `for i in range(max_iterations)` | エッジ（ループ） |
| `if "誤りなし" in critique: break` | 条件付きエッジ (`add_conditional_edges`) |
| `AgentStep` リスト | `AgentState` (TypedDict) |
| `agent.run_pipeline(query)` | `graph.invoke({"query": query})` |


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio langgraph

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
from src.common import load_llm, generate_text, AGENT_MODEL_ID

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


## **1. LangGraph の基本概念**

LangGraph はエージェントの処理フローを **グラフ（ノード＋エッジ）** として記述するライブラリです。

- **ノード (Node)**: 実際の処理（LLM呼び出しなど）を行う Python 関数
- **エッジ (Edge)**: ノード間の遷移（次にどのノードへ進むか）
- **条件付きエッジ (Conditional Edge)**: 状態に応じてノードを分岐させるエッジ
- **State**: ノード間でやり取りされる共有データ（`TypedDict` で型定義）

```
[START] → executor_node
               ↓
         critic_node
               ↓
      ┌── 条件付きエッジ ──┐
"誤りなし"含む           含まない
      ↓                    ↓
   [END]           executor_node (修正)
                          ↓
                    critic_node...
```


In [ ]:
from typing import TypedDict, List, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# ── LLM ラッパー（06回と同じ） ─────────────────────────────────────────
def llm_chat(system_prompt: str, user_prompt: str,
             max_tokens: int = 768, temp: float = 0.5) -> str:
    return generate_text(
        model, tokenizer, user_prompt,
        max_new_tokens=max_tokens, temperature=temp,
        system_prompt=system_prompt
    )

# ── State 定義 ─────────────────────────────────────────────────────────
# TypedDict でノード間を流れるデータを型付きで定義する
# 06回の AgentStep リストに相当するが、LangGraph では State として明示する
class AgentState(TypedDict):
    query: str              # ユーザーの元の質問（変化しない）
    current_answer: str     # Executor の最新の回答
    critique: str           # Critic の最新のフィードバック
    iteration: int          # 現在の反復回数
    max_iterations: int     # 最大反復回数
    # Annotated[List, operator.add] → ノードが返した steps を「追記」する
    steps: Annotated[List[dict], operator.add]

print('State の定義完了')


## **2. ノード関数の定義**

各ノードは `state` を受け取り，更新したいフィールドだけを辞書で返します。
LangGraph が自動的に State をマージします。

06回との比較：
- `LLMExecutorCriticAgent.run_pipeline()` の中に書かれていたロジックを，**executor_node / critic_node** という独立した関数に分割します。
- ループの「繰り返し」はノード間のエッジで表現するため，コード中に `for` ループは登場しません。


In [ ]:
# ── システムプロンプト（06回と同じ内容） ───────────────────────────────
WRITER_SYSTEM_PROMPT = """
あなたはプロのライターです。ユーザーからのテーマについて、まずは標準的な解説記事を書いてください。
記事は必ず「解説」と「具体的な活用シーン」の両方を含めてください。
解説では、仕組み・特徴・社会への影響などを説明し、その後に読者がイメージできる具体的なシーンを示してください。
Editor（編集長）から修正指示が来た場合は、そのフィードバックを全面的に取り入れ、前回の文章から明確に改善された文章を書き直してください。
同じ内容の再提出は禁止です。
"""

EDITOR_SYSTEM_PROMPT = """
あなたは非常に厳しい編集長です。
提出された文章を読み、以下の3つの基準が【すべて】満たされているか評価してください。
1つ目は、テーマについての解説（仕組み・特徴・影響など）が明確に含まれていることです。
2つ目は、読者が日常生活でイメージしやすい具体的な活用シーンが含まれていることです。
3つ目は、全体として読者が体験してみたいと感じるような、ワクワクする感情豊かなトーンになっていることです。
もし、1つでも満たしていない場合は、不足している点を具体的に指摘して書き直しを要求してください。
【重要】絶対に自分で文章を書き直さず、Writerへの修正指示だけを簡潔に出力してください。
すべて満たされていると判断した場合のみ、「誤りなし」と出力してください。
"""

# ── ノード関数 ─────────────────────────────────────────────────────────
# 各ノードは state を受け取り、更新するフィールドのみを dict で返す

def executor_node(state: AgentState) -> dict:
    """Writer: 初稿または修正稿を生成するノード"""
    query = state["query"]
    critique = state.get("critique", "")
    current_answer = state.get("current_answer", "")
    iteration = state.get("iteration", 0)

    if iteration == 0:
        # 初回は質問をそのまま渡す
        user_prompt = query
        role_label = "Writer (初稿)"
    else:
        # 2回目以降は批評を受けて修正
        user_prompt = (
            f"以下の批評を参考に、回答を修正してください。修正後の回答のみを出力してください。\n\n"
            f"【元の質問】\n{query}\n\n"
            f"【現在の回答】\n{current_answer}\n\n【批評】\n{critique}"
        )
        role_label = f"Writer (修正 {iteration})"

    answer = llm_chat(WRITER_SYSTEM_PROMPT, user_prompt)
    print(f"[{role_label}] 生成完了 ({len(answer)}字)")

    return {
        "current_answer": answer,
        "steps": [{"role": role_label, "observation": answer}],
    }


def critic_node(state: AgentState) -> dict:
    """Editor: 回答を批評し、合否を判定するノード"""
    query = state["query"]
    current_answer = state["current_answer"]
    iteration = state.get("iteration", 0)

    critic_input = (
        f"以下の回答をレビューし、システムプロンプトの指示に従ってフィードバックを出力してください。\n\n"
        f"【元の質問】\n{query}\n\n【回答】\n{current_answer}"
    )
    critique = llm_chat(EDITOR_SYSTEM_PROMPT, critic_input)
    role_label = f"Editor (round {iteration + 1})"
    print(f"[{role_label}] 判定: {'✅ 承認' if '誤りなし' in critique else '🔄 修正要求'}")

    return {
        "critique": critique,
        "iteration": iteration + 1,
        "steps": [{"role": role_label, "observation": critique}],
    }

print('ノード関数の定義完了')


## **3. グラフの構築**

ノードをつなぐエッジを定義してグラフを組み立てます。

**ポイント: 条件付きエッジ**

`add_conditional_edges` は「どのノードへ進むか」を関数で決定します。
06回の `if "誤りなし" in critique: break` という条件がここに相当します。


In [ ]:
# ── ルーティング関数 ──────────────────────────────────────────────────
# critic_node の後に呼ばれ、次のノードを文字列で返す
# 06回: if "誤りなし" in critique: break  ← これと等価
def should_continue(state: AgentState) -> str:
    """'誤りなし' が含まれているか、最大反復に達したら END"""
    if "誤りなし" in state["critique"]:
        return "approved"  # → END
    if state["iteration"] >= state["max_iterations"]:
        return "max_reached"  # → END (上限到達)
    return "needs_revision"   # → executor_node へ戻る

# ── グラフ定義 ────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# ノードを登録
builder.add_node("executor", executor_node)
builder.add_node("critic",   critic_node)

# エッジ（処理の流れ）を定義
builder.add_edge(START,      "executor")  # 開始 → executor
builder.add_edge("executor", "critic")    # executor → critic（常に）

# critic → 条件分岐
builder.add_conditional_edges(
    "critic",
    should_continue,
    {
        "approved":      END,        # 承認 → 終了
        "max_reached":   END,        # 上限 → 終了
        "needs_revision": "executor" # 修正要求 → executor に戻る
    }
)

# グラフをコンパイル
graph = builder.compile()
print('グラフのコンパイル完了')


## **4. グラフの可視化（オプション）**

LangGraph はグラフ構造を Mermaid 図として出力できます。
ループや分岐がどのようなグラフになるか確認してみましょう。


In [ ]:
# グラフの Mermaid 図を表示（IPython 環境で動作）
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # 画像生成ライブラリがなければテキストで表示
    print(graph.get_graph().draw_mermaid())


## **5. エージェントの実行**

06回と同じプロンプトで実行し、ステップの流れを確認します。

`graph.invoke()` は全ステップが終わった後の最終 State を返します。
途中状態を逐次確認したい場合は `graph.stream()` を使います（後述）。


In [ ]:
# ── invoke() で一括実行 ───────────────────────────────────────────────
query = "「完全自動運転タクシー」が普及した社会について、300字程度で解説記事を書いてください。"

initial_state: AgentState = {
    "query": query,
    "current_answer": "",
    "critique": "",
    "iteration": 0,
    "max_iterations": 2,  # 06回と同じ上限
    "steps": [],
}

final_state = graph.invoke(initial_state)

print("=" * 60)
print("【最終回答】")
print(final_state["current_answer"])
print("=" * 60)
print(f"\n合計ステップ数: {len(final_state['steps'])}")


## **6. stream() でリアルタイム確認**

`graph.stream()` を使うと，各ノードが完了するたびに State の差分が返ってきます。
これは **06回では実現できなかった機能** です：処理の途中経過をリアルタイムに確認できます。

次のセクション7では，この `stream()` を Gradio と組み合わせ，
**Writer が書き終わるたびに画面が更新されるストリーミング UI** を実装します。


In [ ]:
# ── stream() で逐次出力 ───────────────────────────────────────────────
print("ストリーミング実行開始\n")

for chunk in graph.stream(initial_state):
    # chunk は {ノード名: 更新された State の差分} の辞書
    node_name = list(chunk.keys())[0]
    node_output = chunk[node_name]

    # 直近のステップだけ表示
    if "steps" in node_output and node_output["steps"]:
        latest_step = node_output["steps"][-1]
        print(f"--- [{latest_step['role']}] ---")
        print(latest_step["observation"][:200], "..." if len(latest_step["observation"]) > 200 else "")
        print()


## **7. Gradio ストリーミング UI**

06回の `create_agent_ui()` は `graph.invoke()` で全処理が終わってから結果を返す**バッチ型**でした。

LangGraph の `graph.stream()` と Gradio の**ジェネレーター関数**を組み合わせると，
各ノードが完了するたびに UI が更新される**ストリーミング型**の UI を作れます。

### 06回との実装上の違い

| 比較軸 | 06回 (`invoke` + `return`) | 07回 (`stream` + `yield`) |
|---|---|---|
| **結果の返し方** | 最後に `return` で1回 | ノード完了ごとに `yield` |
| **ユーザー体験** | 全部終わるまで画面が変わらない | Writer/Editor の結果が逐次表示される |
| **Gradio 設定** | 通常の `.click()` | ジェネレーター関数を渡すだけで OK |

### ポイント

Gradio はハンドラー関数が `yield` を使うジェネレーターであることを自動検出し，
`yield` のたびに UI コンポーネントを更新します。特別な設定は不要です。


In [ ]:
import gradio as gr

# ── ストリーミングハンドラー（ジェネレーター関数） ───────────────────────
# graph.stream() でノードが完了するたびに yield → Gradio が UI を更新する
def run_streaming_agent(query: str):
    """各ノード完了時に (最終回答候補, ログ累計) を yield するジェネレーター"""
    state: AgentState = {
        "query": query,
        "current_answer": "",
        "critique": "",
        "iteration": 0,
        "max_iterations": 2,
        "steps": [],
    }

    log_so_far = ""
    current_answer = "（処理中…）"

    # graph.stream() は各ノード完了時に {ノード名: 更新差分} を yield する
    for chunk in graph.stream(state):
        node_name = list(chunk.keys())[0]   # 'executor' or 'critic'
        node_output = chunk[node_name]

        # steps に追記されたログを取り出す
        if "steps" in node_output and node_output["steps"]:
            step = node_output["steps"][-1]
            role = step["role"]
            obs  = step["observation"]
            # Markdown の引用形式でログを蓄積
            log_so_far += f"### {role}\n> {obs.replace(chr(10), chr(10) + '> ')}\n\n"

        # Executor のノードが完了したら「現在の最善回答」を更新
        if node_name == "executor" and "current_answer" in node_output:
            current_answer = node_output["current_answer"]

        # ← このノードの出力を Gradio へ即座に送信
        yield current_answer, log_so_far

# ── Gradio UI の構築 ─────────────────────────────────────────────────
# Gradio はハンドラーが yield するジェネレーターであることを自動検出する
with gr.Blocks(title="LangGraph Streaming Agent") as ui:
    gr.Markdown("# LangGraph ストリーミング Executor-Critic Agent")
    gr.Markdown(
        "Writer と Editor の処理が完了するたびに、画面がリアルタイムで更新されます。"
    )

    with gr.Row():
        with gr.Column(scale=2):
            query_box = gr.Textbox(
                label="ユーザーの質問",
                placeholder="例: 「完全自動運転タクシー」が普及した社会について、300字程度で解説記事を書いてください。",
                lines=3,
            )
            submit_btn = gr.Button("エージェントに依頼", variant="primary")

        with gr.Column(scale=3):
            # ノード完了ごとに更新される出力コンポーネント
            answer_box = gr.Textbox(
                label="最終回答（Writer が完了するたびに更新）",
                lines=10,
            )

    with gr.Accordion("思考プロセスの詳細ログ（ノード完了ごとに追記）", open=True):
        log_box = gr.Markdown(label="ログ")

    # streaming=True は不要: ジェネレーターを渡すだけで Gradio が自動処理する
    submit_btn.click(
        fn=run_streaming_agent,
        inputs=[query_box],
        outputs=[answer_box, log_box],
    )

ui.launch(share=True, debug=True)


## **まとめ**

### 06回（手作りループ）と 07回（LangGraph + Streaming）の比較

| 比較軸 | 06回 | 07回 (LangGraph) |
|---|---|---|
| **ループ制御** | `for` + `break` | グラフのエッジで宣言的に |
| **状態管理** | 変数の引き回し | `TypedDict` で型付き |
| **分岐条件** | `if 誤りなし in critique` | `add_conditional_edges` |
| **途中状態の取得** | 非対応 | `stream()` で実現 |
| **Gradio UI** | `return` で1回だけ更新 | `yield` でノード完了ごとに更新 |
| **コード量** | 少ない | 多い（ノード・エッジ定義が増える） |
| **拡張性** | ノード追加にコード改変が必要 | ノード登録だけで追加できる |

### どちらを使うべきか

- **シンプルな 2エージェント・逐次処理** → 06回のような手作りループで十分
- **ストリーミング表示・複数分岐・Human-in-the-loop** を入れたい → LangGraph が有効

LangGraph は「グラフとして表現することでシステムの全体像が一目でわかる」という設計上のメリットがあります。
また `stream()` + Gradio ジェネレーターの組み合わせにより，
**「エージェントが考えている途中」をリアルタイムで見せる** ことが可能になります。
チームでエージェントを開発する場面では，可読性・拡張性の観点から導入を検討する価値があります。
